In [1]:
!pip install playwright nest-asyncio pandas openpyxl playwright-stealth
!playwright install chromium firefox webkit

source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory



In [2]:
# Cell login thủ công — KHÔNG CẦN CHẠY
# Crawl vẫn hoạt động bình thường mà không cần cookies.
# Chỉ chạy cell này nếu Trip.com yêu cầu đăng nhập mới xem giá.

### Main code

In [3]:
# ============================================================
# CRAWL Trip.com — v3.1 (API-first: nhanh & chính xác)
# ============================================================
# Luồng: đọc list KS từ CSV (Hotel, URL, Room) -> mở trang Trip.com ->
#   BẮT THẲNG JSON API `getHotelRoomList` (thay vì scrape DOM) -> lấy giá
#   "Total incl. taxes & fees" của đúng room type -> 6 tuần.
#
# FIX v3.1 (vì sao trước đây NA rất nhiều):
#   1) GIỮ NGUYÊN host của URL (vn/www/ca.trip.com). API trả tên phòng theo
#      NGÔN NGỮ của host: vn.trip.com -> tiếng Việt, www/ca -> tiếng Anh.
#      Bản v3 ép tất cả về www.trip.com + locale=en-XX -> tên phòng ra tiếng Anh,
#      trong khi room_type trong CSV là tiếng Việt -> token match = 0 -> NA hàng loạt.
#   2) Trip.com tải list phòng LAZY: response getHotelRoomList ĐẦU TIÊN là skeleton
#      RỖNG (physicRoomMap={}); response CÓ PHÒNG là POST tới SAU, chỉ kích hoạt khi
#      CUỘN trang. Bản v3 lấy response đầu tiên rồi dừng -> toàn rỗng -> NA.
#      v3.1: CUỘN trang + chỉ nhận response CÓ physicRoomMap/saleRoomMap.
#
# Vì sao nhanh hơn code cũ:
#   • Chờ ĐÚNG response API (không sleep/scroll mò) + chặn ảnh/media/font
#   • Crawl 6 tuần SONG SONG trong mỗi khách sạn
# Vì sao chính xác hơn:
#   • Dữ liệu JSON có cấu trúc (tên phòng + giá) thay vì regex trên DOM
#   • Match phòng theo TOKEN (không còn substring 2 chiều dễ khớp nhầm)
#   • Lấy đúng "Total incl. taxes & fees", chọn rate plan rẻ nhất của phòng

import asyncio
import nest_asyncio
import pandas as pd
import random
import time
import os
import re
import json
import glob
from datetime import datetime, timedelta
from playwright.async_api import async_playwright, TimeoutError as PWTimeout
from playwright_stealth import Stealth
from openpyxl import load_workbook

nest_asyncio.apply()
_stealth = Stealth()

# ============================================================
# CONFIG
# ============================================================
# Tự tìm file input trip*.csv trong thư mục hiện tại (bỏ qua TEMP_/FINAL_)
_csvs = sorted(f for f in glob.glob("*.csv") if not f.startswith(("TEMP_", "FINAL_")))
INPUT_FILE = _csvs[0] if _csvs else "./trip1.csv"
TEMP_OUTPUT_FILE = "TEMP_" + os.path.basename(INPUT_FILE)
OUTPUT_PREFIX = "FINAL_"
INPUT_SHEET_NAME = "Hotel Link"        # chỉ dùng cho .xlsx
GOOGLE_SHEET_ID = ""                    # để trống = đọc file local
GOOGLE_SHEET_NAME = "Hotel Link"
COOKIES_FILE = "cookies.json"          # tuỳ chọn, không bắt buộc

# Loại giá ghi ra: total_incl_tax (đã chọn) | display | original
PRICE_TYPE = "total_incl_tax"

HEADLESS = True
CHECKIN_OFFSET = 3                      # W1 = hôm nay + 3 ngày
NUM_WEEKS = 6
DAYS_PER_WEEK = 3                       # số ngày thử/tuần (dừng khi có giá)
WEEKS_PARALLEL = 6                      # số tuần crawl song song trong 1 KS
PAGE_TIMEOUT = 35000
API_WAIT_TIMEOUT = 22                   # giây: chờ response getHotelRoomList (CÓ phòng)
CURRENCY = "VND"

BETWEEN_HOTELS = (1.5, 4.0)            # nghỉ giữa các khách sạn
NAV_JITTER = (0.3, 1.2)               # jitter nhỏ trước mỗi lần mở trang
INTRA_WEEK_DELAY = (0.4, 1.2)         # nghỉ giữa các ngày trong 1 tuần
AUTO_RETRY_NA_SOLDOUT = True          # vòng 2: re-crawl NA/SOLD OUT
RETRY_DAYS_PER_WEEK = 5               # vòng 2 thử nhiều ngày hơn

ROOM_API_HINT = "getHotelRoomList"    # endpoint chứa list phòng + giá
BLOCK_RESOURCE_TYPES = {"image", "media", "font"}

USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]
SCREEN_RESOLUTIONS = [(1366, 768), (1440, 900), (1536, 864), (1600, 900), (1920, 1080), (1680, 1050)]

# ============================================================
# EXTRACTION (đã test với dữ liệu Trip.com thật)
# ============================================================
def _to_int(s):
    n = re.sub(r'[^\d]', '', s or '')
    return int(n) if n else None

def parse_vnd(text):
    if not text:
        return None
    m = re.search(r'VND\s*([\d.,]+)', str(text), re.I)
    if not m:
        m = re.search(r'([\d][\d.,]{3,})', str(text))
    if not m:
        return None
    v = _to_int(m.group(1))
    return v if v and v >= 1000 else None

def fmt_vnd(v):
    return f"VND {v:,}"

def _norm(s):
    return re.sub(r'[^a-z0-9]+', ' ', (s or '').lower()).strip()

def _tokens(s):
    return set(t for t in _norm(s).split() if t)

def pick_room(name2prices, target):
    """Chọn phòng khớp nhất với target -> (name, giá thấp nhất). Token-based."""
    if not name2prices:
        return None, None
    tnorm = _norm(target)
    ttok = _tokens(target)
    for name, prices in name2prices.items():            # 1) khớp chính xác
        if _norm(name) == tnorm and prices:
            return name, min(prices)
    best, best_score = None, -1.0                        # 2) khớp theo độ phủ token
    for name, prices in name2prices.items():
        if not prices:
            continue
        ntok = _tokens(name)
        if not ntok:
            continue
        score = len(ttok & ntok) / max(len(ttok), 1)
        if ttok <= ntok or ntok <= ttok:
            score += 0.5
        if score > best_score:
            best_score, best = score, (name, min(prices))
    return best if (best and best_score >= 0.5) else (None, None)

def sale_price(pinfo, ptype=PRICE_TYPE):
    pinfo = pinfo or {}
    if ptype == "total_incl_tax":
        return parse_vnd(pinfo.get("priceExplanation")) or parse_vnd(pinfo.get("displayPrice"))
    if ptype == "display":
        v = parse_vnd(pinfo.get("displayPrice"))
        if v:
            return v
        p = pinfo.get("price")
        return int(p) if isinstance(p, (int, float)) and p >= 1000 else None
    if ptype == "original":
        p = pinfo.get("deletePricewithOutCurrency")
        if isinstance(p, (int, float)) and p >= 1000:
            return int(p)
        return parse_vnd(pinfo.get("deletePrice"))
    return None

def extract_from_api(api, target_room, ptype=PRICE_TYPE):
    d = (api or {}).get("data") or {}
    phys = d.get("physicRoomMap") or {}
    sales = d.get("saleRoomMap") or {}
    id2name = {str(k): (v.get("name") or "").strip() for k, v in phys.items()}
    name2prices = {}
    for sid, sr in sales.items():
        pid = str(sr.get("physicalRoomId") or "")
        name = id2name.get(pid) or (sr.get("name") or "").strip()
        if not name:
            continue
        p = sale_price(sr.get("priceInfo") or {}, ptype)
        if p:
            name2prices.setdefault(name, []).append(p)
    name, price = pick_room(name2prices, target_room)
    if price:
        return {"found": True, "price": fmt_vnd(price), "room": name}
    if d.get("isRoomListSoldOut"):
        return {"found": False, "soldOut": True}
    return {"found": False, "soldOut": False, "rooms": list(name2prices.keys())}

# DOM fallback (chỉ dùng khi KHÔNG bắt được API)
EXTRACT_DOM_JS = r"""() => {
  const cards = document.querySelectorAll("div[class*='commonRoomCard__'], div[class*='saleRoomItemBox__']");
  const rooms = [];
  cards.forEach(c => {
    const t = c.querySelector("span[class*='commonRoomCard-title'], [class*='saleRoomItemBox-head-title'], h2");
    rooms.push({name: t ? t.textContent.trim() : '', text: (c.innerText || '')});
  });
  const body = document.body.innerText || '';
  return {rooms, soldOut: /sold\s*out|hết\s*phòng/i.test(body) && cards.length === 0};
}"""

def extract_from_dom(dom, target_room, ptype=PRICE_TYPE):
    name2prices = {}
    for r in (dom.get("rooms") or []):
        name = (r.get("name") or "").strip()
        if not name:
            continue
        text = r.get("text") or ""
        v = None
        if ptype == "total_incl_tax":
            m = re.search(r'Total\s*\(incl[^)]*\)\s*:?\s*(VND\s*[\d.,]+)', text, re.I)
            if not m:
                m = re.search(r'Tổng\s*giá[^\d]*([\d.,]+)', text, re.I)   # nhãn tiếng Việt
            if m:
                v = parse_vnd(m.group(1))
        if v is None:
            v = parse_vnd(text)
        if v:
            name2prices.setdefault(name, []).append(v)
    name, price = pick_room(name2prices, target_room)
    if price:
        return {"found": True, "price": fmt_vnd(price), "room": name}
    if dom.get("soldOut"):
        return {"found": False, "soldOut": True}
    return {"found": False, "soldOut": False, "rooms": list(name2prices.keys())}

def is_real(v):
    return v not in (None, "", "NA", "nan") and not str(v).startswith("SOLD OUT")

# ============================================================
# INPUT READERS
# ============================================================
def read_hotels_from_source():
    if GOOGLE_SHEET_ID:
        return read_hotels_from_gsheet(GOOGLE_SHEET_ID, GOOGLE_SHEET_NAME)
    if INPUT_FILE.endswith('.xlsx'):
        return read_hotels_from_xlsx(INPUT_FILE, INPUT_SHEET_NAME)
    return read_hotels_from_csv(INPUT_FILE)

def _norm_cols(df):
    # GIỮ NGUYÊN host của URL (vn/www/ca.trip.com) -> tên phòng đúng ngôn ngữ với room_type.
    req = ['hotel_name', 'hotel_url', 'room_type']
    if not all(c in df.columns for c in req):
        df.columns = req + list(df.columns[3:])
    df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
    return df[['hotel_name', 'hotel_url', 'room_type']]

def read_hotels_from_csv(file_path):
    try:
        df = _norm_cols(pd.read_csv(file_path))
        print(f"✅ {len(df)} hotels từ {file_path}", flush=True)
        return df
    except Exception as e:
        print(f"❌ Lỗi đọc CSV: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_xlsx(file_path, sheet_name):
    try:
        wb = load_workbook(filename=file_path, data_only=True)
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows(min_row=2, max_col=2):
            hcell, rcell = row[0], row[1]
            url = hcell.hyperlink.target if hcell.hyperlink else (hcell.value or "")
            rows.append({'hotel_name': hcell.value or "", 'hotel_url': url,
                         'room_type': (rcell.value if rcell else "")})
        df = pd.DataFrame(rows)
        df = df[df['hotel_url'].notna() & df['hotel_url'].astype(str).str.startswith('http')]
        print(f"✅ {len(df)} hotels từ {file_path}", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"❌ Lỗi đọc xlsx: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_gsheet(sheet_id, sheet_name=""):
    from urllib.parse import quote
    try:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv"
        if sheet_name:
            url += f"&sheet={quote(sheet_name)}"
        df = _norm_cols(pd.read_csv(url))
        print(f"✅ {len(df)} hotels từ Google Sheet", flush=True)
        return df
    except Exception as e:
        print(f"❌ Lỗi đọc Google Sheet: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

# ============================================================
# URL & SAVE
# ============================================================
def update_url_checkin(url, checkin):
    # GIỮ NGUYÊN host (vn/www/ca.trip.com) + locale gốc: tên phòng trả về theo NGÔN NGỮ
    # của host nên khớp đúng room_type trong CSV. Chỉ set checkIn/checkOut + curr.
    ci = checkin.strftime("%Y-%m-%d")
    co = (checkin + timedelta(days=1)).strftime("%Y-%m-%d")
    url = re.sub(r'check[Ii]n=[\d-]+', f'checkIn={ci}', url) if re.search(r'check[Ii]n=', url) \
        else url + f"{'&' if '?' in url else '?'}checkIn={ci}"
    url = re.sub(r'check[Oo]ut=[\d-]+', f'checkOut={co}', url) if re.search(r'check[Oo]ut=', url) \
        else url + f"&checkOut={co}"
    if 'curr=' not in url.lower():
        url += f'&curr={CURRENCY}'
    return url

def save_backup_csv(all_week_prices, filename):
    """Lưu CSV — only-improve (không ghi NA đè giá thật) + atomic write."""
    try:
        rows, written = [], set()
        if os.path.exists(filename):
            try:
                df_old = pd.read_csv(filename, keep_default_na=False, na_values=[])
                for _, row in df_old.iterrows():
                    k = (str(row.get("hotel_name", "")), str(row.get("room_type", "")))
                    written.add(k)
                    if k in all_week_prices:
                        nr = {"hotel_name": k[0], "room_type": k[1]}
                        for i in range(1, NUM_WEEKS + 1):
                            old = str(row.get(f"price_w{i}", "NA")).strip()
                            new = str(all_week_prices[k].get(f"Price W{i}", "NA")).strip()
                            if new in ("NA", "nan", "") and old not in ("NA", "nan", ""):
                                nr[f"price_w{i}"] = old
                            else:
                                nr[f"price_w{i}"] = new if new not in ("nan", "") else "NA"
                        rows.append(nr)
                    else:
                        rows.append(row.to_dict())
            except Exception:
                pass
        for (hotel, room), prices in all_week_prices.items():
            if (hotel, room) not in written:
                r = {"hotel_name": hotel, "room_type": room}
                for i in range(1, NUM_WEEKS + 1):
                    r[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
                rows.append(r)
        tmp = filename + ".tmp"
        pd.DataFrame(rows).to_csv(tmp, index=False)
        os.replace(tmp, filename)
    except Exception as e:
        print(f"❌ Error saving: {e}", flush=True)

async def load_cookies_to_context(context):
    if not os.path.exists(COOKIES_FILE):
        return
    try:
        with open(COOKIES_FILE, "r", encoding="utf-8") as f:
            cookies = json.load(f)
        pw = []
        for c in cookies:
            ck = {"name": c.get("name", ""), "value": c.get("value", ""),
                  "domain": c.get("domain", ".trip.com"), "path": c.get("path", "/")}
            for opt in ("expires", "httpOnly", "secure"):
                if c.get(opt):
                    ck[opt] = c[opt]
            if c.get("sameSite") in ("Strict", "Lax", "None"):
                ck["sameSite"] = c["sameSite"]
            pw.append(ck)
        await context.add_cookies(pw)
    except Exception:
        pass

# ============================================================
# BROWSER / CRAWL
# ============================================================
async def _block_route(route):
    try:
        if route.request.resource_type in BLOCK_RESOURCE_TYPES:
            await route.abort()
        else:
            await route.continue_()
    except Exception:
        try:
            await route.continue_()
        except Exception:
            pass

async def make_context(browser):
    res = random.choice(SCREEN_RESOLUTIONS)
    ctx = await browser.new_context(
        viewport={"width": res[0], "height": res[1]},
        user_agent=random.choice(USER_AGENTS),
        locale="en-US",
        timezone_id="Asia/Ho_Chi_Minh",
        java_script_enabled=True,
    )
    await ctx.route("**/*", _block_route)
    await load_cookies_to_context(ctx)
    return ctx

async def crawl_day(ctx, hotel_url, room_type, checkin, page_timeout=PAGE_TIMEOUT):
    """Mở 1 ngày check-in, bắt JSON API getHotelRoomList (response CÓ phòng) -> giá.

    Trip.com tải list phòng kiểu LAZY (React Server Component): response ĐẦU TIÊN
    của getHotelRoomList thường là skeleton RỖNG (physicRoomMap={}); response CÓ
    PHÒNG là POST tới SAU, chỉ kích hoạt khi CUỘN tới khu vực phòng. Vì vậy:
      • CUỘN trang để kích hoạt POST list phòng
      • Chỉ nhận response CÓ physicRoomMap/saleRoomMap (bỏ qua skeleton rỗng)
    Fallback DOM nếu không bắt được API."""
    page = await ctx.new_page()
    cap = {}   # cap["api"] = JSON có phòng; cap["soldout"] = JSON sold-out thật

    async def on_resp(resp):
        try:
            if ROOM_API_HINT not in resp.url or resp.status != 200 or cap.get("api"):
                return
            data = (await resp.json()).get("data") or {}
            if data.get("physicRoomMap") or data.get("saleRoomMap"):
                cap["api"] = {"data": data}            # response CÓ phòng -> ưu tiên
            elif data.get("isRoomListSoldOut"):
                cap["soldout"] = {"data": data}        # sold-out thật (map rỗng + cờ bật)
        except Exception:
            pass

    page.on("response", lambda r: asyncio.create_task(on_resp(r)))
    try:
        await _stealth.apply_stealth_async(page)
        url = update_url_checkin(hotel_url, checkin)
        await asyncio.sleep(random.uniform(*NAV_JITTER))
        try:
            await page.goto(url, timeout=page_timeout, wait_until="domcontentloaded")
        except Exception:
            pass

        # CUỘN để kích hoạt POST list phòng (lazy) + chờ response CÓ phòng
        waited = 0.0
        while waited < API_WAIT_TIMEOUT and not cap.get("api"):
            if cap.get("soldout") and waited > 8:      # sold-out thật: không cần chờ thêm
                break
            try:
                await page.mouse.wheel(0, 2500)
            except Exception:
                pass
            await asyncio.sleep(0.6)
            waited += 0.6

        if cap.get("api"):
            try:
                res = extract_from_api(cap["api"], room_type)
                if res.get("found") or res.get("soldOut"):
                    return res
            except Exception:
                pass
        if cap.get("soldout"):
            return {"found": False, "soldOut": True}

        # Fallback DOM (hiếm khi cần) — đã cuộn nên thường đã có card phòng
        try:
            await page.wait_for_selector(
                "div[class*='commonRoomCard__'], div[class*='saleRoomItemBox__']", timeout=6000)
        except Exception:
            pass
        try:
            dom = await page.evaluate(EXTRACT_DOM_JS)
            return extract_from_dom(dom, room_type)
        except Exception:
            return {"found": False, "soldOut": False}
    finally:
        try:
            await page.close()
        except Exception:
            pass

async def crawl_week(ctx, hotel_url, room_type, week_num, base_checkin,
                     hotel_name="", days=DAYS_PER_WEEK, page_timeout=PAGE_TIMEOUT):
    week_start = base_checkin + timedelta(days=(week_num - 1) * 7)
    sold = False
    for d in range(days):
        checkin = week_start + timedelta(days=d)
        res = await crawl_day(ctx, hotel_url, room_type, checkin, page_timeout)
        if res.get("found"):
            print(f"      ✅ [{hotel_name[:22]}] W{week_num}: {res['price']} "
                  f"({checkin.strftime('%m/%d')}, {res.get('room','')[:20]})", flush=True)
            return {"week": week_num, "price": res["price"]}
        if res.get("soldOut"):
            sold = True
        if d < days - 1:
            await asyncio.sleep(random.uniform(*INTRA_WEEK_DELAY))
    price = "SOLD OUT" if sold else "NA"
    print(f"      {'🚫' if sold else '❌'} [{hotel_name[:22]}] W{week_num}: {price}", flush=True)
    return {"week": week_num, "price": price}

async def process_hotel(browser, info, prev, base_checkin, awp):
    hotel_name, hotel_url, room_type = info
    key = (hotel_name, room_type)

    prices = {f"Price W{i}": "NA" for i in range(1, NUM_WEEKS + 1)}
    if key in prev:
        for i in range(1, NUM_WEEKS + 1):
            v = prev[key].get(f"Price W{i}", "NA")
            if is_real(v):
                prices[f"Price W{i}"] = v

    weeks = [i for i in range(1, NUM_WEEKS + 1) if not is_real(prices[f"Price W{i}"])]
    if not weeks:
        awp[key] = prices
        return key, prices, True

    print(f"\n🏨 {hotel_name} | {room_type} | tuần cần crawl: {weeks}", flush=True)
    ctx = await make_context(browser)
    try:
        sem = asyncio.Semaphore(WEEKS_PARALLEL)

        async def one(wn):
            async with sem:
                return await crawl_week(ctx, hotel_url, room_type, wn, base_checkin, hotel_name)

        results = await asyncio.gather(*[one(w) for w in weeks])
    finally:
        try:
            await ctx.close()
        except Exception:
            pass

    for r in results:
        prices[f"Price W{r['week']}"] = r["price"]
    awp[key] = prices
    na = sum(1 for i in range(1, NUM_WEEKS + 1) if prices[f"Price W{i}"] == "NA")
    so = sum(1 for i in range(1, NUM_WEEKS + 1) if str(prices[f"Price W{i}"]).startswith("SOLD OUT"))
    icon = "✅" if na == 0 and so == 0 else (f"⚠️({na}NA)" if na else f"🚫({so}SO)")
    print(f"   {icon} DONE: {hotel_name}", flush=True)
    return key, prices, False

# ============================================================
# MAIN
# ============================================================
async def main():
    t0 = time.time()
    df = read_hotels_from_source()
    if len(df) == 0:
        return

    awp, prev = {}, {}

    def load_prev(fp):
        n = 0
        try:
            dp = pd.read_csv(fp, keep_default_na=False, na_values=[])
            for _, row in dp.iterrows():
                k = (row["hotel_name"], row["room_type"])
                p = {}
                for i in range(1, NUM_WEEKS + 1):
                    v = str(row.get(f"price_w{i}", "NA")).strip()
                    p[f"Price W{i}"] = "NA" if (not v or v in ("nan", "NA")) else v
                prev[k] = p
                awp[k] = dict(p)
                n += 1
        except Exception:
            pass
        return n

    if os.path.exists(TEMP_OUTPUT_FILE):
        print(f"📂 Loaded {load_prev(TEMP_OUTPUT_FILE)} hotels từ temp", flush=True)

    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=CHECKIN_OFFSET)
    all_infos = [(r['hotel_name'], r['hotel_url'], r['room_type']) for _, r in df.iterrows()]

    done = sum(1 for info in all_infos
               if (info[0], info[2]) in prev
               and all(is_real(prev[(info[0], info[2])].get(f"Price W{i}", "NA")) for i in range(1, NUM_WEEKS + 1)))

    print(f"\n📅 W1 = {bc.strftime('%Y-%m-%d (%a)')} ... W{NUM_WEEKS} (mỗi tuần thử {DAYS_PER_WEEK} ngày)", flush=True)
    print(f"{'='*60}", flush=True)
    print(f"🎭 CRAWL Trip.com v3.1 — API-first ({ROOM_API_HINT}) | giá: {PRICE_TYPE}", flush=True)
    print(f"📊 {len(all_infos)} hotels ({done} đã đủ 6w) | {WEEKS_PARALLEL} tuần song song/KS", flush=True)
    print(f"{'='*60}", flush=True)

    launch_args = ['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-dev-shm-usage']
    if HEADLESS:
        launch_args.insert(0, '--headless=new')

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS, args=launch_args)
        try:
            for idx, info in enumerate(all_infos, 1):
                try:
                    key, prices, skip = await process_hotel(browser, info, prev, bc, awp)
                    if not skip:
                        save_backup_csv(awp, TEMP_OUTPUT_FILE)
                        await asyncio.sleep(random.uniform(*BETWEEN_HOTELS))
                except Exception as e:
                    print(f"❌ {info[0]}: {e}", flush=True)
                print(f"   ⏱️ {idx}/{len(all_infos)} | {int((time.time()-t0)//60)}m", flush=True)

            fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
            save_backup_csv(awp, fn)
            tc = len(awp) * NUM_WEEKS
            na = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if pp.get(f"Price W{i}", "NA") == "NA")
            so = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if str(pp.get(f"Price W{i}", "")).startswith("SOLD OUT"))
            print(f"\n📁 Round 1: {fn} | ✅ {tc-na-so}/{tc} | 🚫 {so} SO | ❌ {na} NA", flush=True)

            # ----- VÒNG 2: re-crawl NA / SOLD OUT -----
            if AUTO_RETRY_NA_SOLDOUT and (na > 0 or so > 0):
                ki = {(i[0], i[2]): i for i in all_infos}
                retry = {k: [i for i in range(1, NUM_WEEKS + 1)
                             if not is_real(awp[k].get(f"Price W{i}", "NA"))]
                         for k in awp if any(not is_real(awp[k].get(f"Price W{i}", "NA")) for i in range(1, NUM_WEEKS + 1))}
                retry = {k: v for k, v in retry.items() if v and k in ki}
                print(f"\n🔄 VÒNG 2: {sum(len(v) for v in retry.values())} ô / {len(retry)} KS", flush=True)
                updated = 0
                for k, weeks in retry.items():
                    hn, hu, rt = ki[k]
                    ctx = await make_context(browser)
                    try:
                        sem = asyncio.Semaphore(WEEKS_PARALLEL)

                        async def one(wn):
                            async with sem:
                                return await crawl_week(ctx, hu, rt, wn, bc, hn,
                                                        days=RETRY_DAYS_PER_WEEK, page_timeout=45000)
                        res = await asyncio.gather(*[one(w) for w in weeks])
                    finally:
                        try:
                            await ctx.close()
                        except Exception:
                            pass
                    for r in res:
                        old = awp[k].get(f"Price W{r['week']}", "NA")
                        new = r["price"]
                        if is_real(new) and new != old:
                            awp[k][f"Price W{r['week']}"] = new
                            updated += 1
                        elif new == "SOLD OUT" and old == "NA":
                            awp[k][f"Price W{r['week']}"] = new
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                    await asyncio.sleep(random.uniform(*BETWEEN_HOTELS))
                save_backup_csv(awp, fn)
                print(f"   🔄 Vòng 2 xong: cập nhật {updated} ô", flush=True)
        finally:
            try:
                await browser.close()
            except Exception:
                pass

    tt = time.time() - t0
    tc = len(awp) * NUM_WEEKS
    na = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if pp.get(f"Price W{i}", "NA") == "NA")
    so = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if str(pp.get(f"Price W{i}", "")).startswith("SOLD OUT"))
    fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
    print(f"\n{'='*60}", flush=True)
    print(f"✅ HOÀN TẤT | {fn}", flush=True)
    print(f"   ✅ Giá: {tc-na-so}/{tc} ({(tc-na-so)/max(tc,1):.1%}) | 🚫 {so} SO | ❌ {na} NA", flush=True)
    print(f"⏱️ {int(tt//60)}m {int(tt%60)}s", flush=True)
    print(f"{'='*60}", flush=True)

await main()

✅ 16 hotels từ trip2.csv
📂 Loaded 16 hotels từ temp

📅 W1 = 2026-06-27 (Sat) ... W6 (mỗi tuần thử 3 ngày)
🎭 CRAWL Trip.com v3.1 — API-first (getHotelRoomList) | giá: total_incl_tax
📊 16 hotels (10 đã đủ 6w) | 6 tuần song song/KS
📂 Loaded 16 hotels từ temp

📅 W1 = 2026-06-27 (Sat) ... W6 (mỗi tuần thử 3 ngày)
🎭 CRAWL Trip.com v3.1 — API-first (getHotelRoomList) | giá: total_incl_tax
📊 16 hotels (10 đã đủ 6w) | 6 tuần song song/KS
   ⏱️ 1/16 | 0m
   ⏱️ 2/16 | 0m
   ⏱️ 3/16 | 0m
   ⏱️ 4/16 | 0m
   ⏱️ 5/16 | 0m
   ⏱️ 6/16 | 0m

🏨 PARKROYAL Serviced Suites Hanoi | Phòng Deluxe có 1 Phòng ngủ | tuần cần crawl: [1, 2, 3, 4, 5, 6]
      ✅ [PARKROYAL Serviced Sui] W6: VND 2,570,764 (08/01, Phòng Deluxe có 1 Ph)
      ✅ [PARKROYAL Serviced Sui] W5: VND 2,570,764 (07/25, Phòng Deluxe có 1 Ph)
      ✅ [PARKROYAL Serviced Sui] W1: VND 2,537,852 (06/27, Phòng Deluxe có 1 Ph)
      ✅ [PARKROYAL Serviced Sui] W4: VND 2,570,764 (07/18, Phòng Deluxe có 1 Ph)
      ✅ [PARKROYAL Serviced Sui] W3: VND 2,57